# A Minimal Decoder-Only Transformer

The original Transformer pairs an encoder with a decoder. For continuation-style language modeling, a decoder-only architecture removes the encoder and cross-attention, then predicts each next token using causal self-attention over the tokens that came before it.

This CPU-only notebook assembles the essential pieces into one small block: token embeddings, fixed sinusoidal positions, causal multi-head self-attention, residual connections, LayerNorm, a feed-forward sublayer, and next-token logits. A single deterministic forward and backward pass checks the tensor shapes and gradient path. It does not train a language model.

**Design reference:** Vaswani et al., [*Attention Is All You Need*](https://arxiv.org/abs/1706.03762). This compact implementation retains the paper's core attention, positional-information, and decoder components while omitting the encoder, cross-attention, dropout, layer stacking, and training.

## 1. Add position and compute causal multi-head attention

Token embeddings identify vocabulary items but do not by themselves identify token order. The original Transformer adds fixed sine and cosine position vectors to the embeddings. The attention module then projects each representation into queries, keys, and values for several heads. Each head uses a causal mask, so a query can use its own and earlier positions but never a later one.

In [1]:
import math

import torch
from torch import nn

torch.manual_seed(7)
VOCABULARY = (
    '<bos>', 'The', 'animal', "didn't", 'cross', 'the', 'street',
    'because', 'it', 'was', 'too', 'tired', '<eos>',
)
TOKEN_TO_ID = {token: index for index, token in enumerate(VOCABULARY)}
ID_TO_TOKEN = dict(enumerate(VOCABULARY))
VOCAB_SIZE, CONTEXT, D_MODEL, HEADS = len(VOCABULARY), 12, 24, 4
assert D_MODEL % HEADS == 0


In [2]:
def sinusoidal_positions(length, width):
    positions = torch.arange(length).unsqueeze(1)
    frequencies = torch.exp(torch.arange(0, width, 2) * (-math.log(10_000.0) / width))
    values = torch.zeros(length, width)
    values[:, 0::2] = torch.sin(positions * frequencies)
    values[:, 1::2] = torch.cos(positions * frequencies)
    return values


class CausalMultiHeadAttention(nn.Module):
    def __init__(self, width, heads):
        super().__init__()
        self.heads = heads
        self.head_width = width // heads
        self.qkv = nn.Linear(width, 3 * width, bias=False)
        self.output = nn.Linear(width, width, bias=False)

    def forward(self, x):
        batch, tokens, width = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)

        def split_heads(tensor):
            return tensor.view(batch, tokens, self.heads, self.head_width).transpose(1, 2)

        q, k, v = map(split_heads, (q, k, v))
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.head_width)
        causal_mask = torch.ones(tokens, tokens, dtype=torch.bool, device=x.device).tril()
        weights = torch.softmax(scores.masked_fill(~causal_mask, float("-inf")), dim=-1)
        context = (weights @ v).transpose(1, 2).contiguous().view(batch, tokens, width)
        return self.output(context), weights


## 2. Form one decoder block

The block below uses the original Transformer-style **post-norm** order: each sublayer output is added through a residual connection and then normalized. A decoder-only block has causal self-attention and no encoder-decoder cross-attention. The final linear layer maps each contextual representation to one logit for every vocabulary item.

In [3]:
class DecoderBlock(nn.Module):
    def __init__(self, width, heads, expansion=4):
        super().__init__()
        self.attention = CausalMultiHeadAttention(width, heads)
        self.attention_norm = nn.LayerNorm(width)
        self.mlp = nn.Sequential(
            nn.Linear(width, expansion * width),
            nn.GELU(),
            nn.Linear(expansion * width, width),
        )
        self.mlp_norm = nn.LayerNorm(width)

    def forward(self, x):
        attended, weights = self.attention(x)
        x = self.attention_norm(x + attended)
        x = self.mlp_norm(x + self.mlp(x))
        return x, weights


class TinyDecoderLM(nn.Module):
    def __init__(self, vocabulary_size, context, width, heads):
        super().__init__()
        self.token_embedding = nn.Embedding(vocabulary_size, width)
        self.register_buffer("positions", sinusoidal_positions(context, width), persistent=False)
        self.block = DecoderBlock(width, heads)
        self.output = nn.Linear(width, vocabulary_size, bias=False)

    def forward(self, token_ids):
        x = self.token_embedding(token_ids) + self.positions[: token_ids.size(1)]
        x, weights = self.block(x)
        return self.output(x), weights


## 3. Create shifted next-token targets

One token sequence supplies both inputs and labels. At each input position, the target is the following token. This one-position shift is the next-token-prediction objective used for an autoregressive language model.

### Check the shifted tensors

The small vocabulary maps the running sentence to readable IDs. The assertions below confirm that inputs and targets have the same shape and that each target aligns with the next input token.

In [4]:
sequence_tokens = VOCABULARY
sequence = torch.tensor([[TOKEN_TO_ID[token] for token in sequence_tokens]], dtype=torch.long)
token_ids = sequence[:, :-1]
targets = sequence[:, 1:]

def decode_token_ids(ids):
    return " ".join(ID_TO_TOKEN[token_id] for token_id in ids)

print("input tokens:       ", decode_token_ids(token_ids[0].tolist()))
print("next-token targets:", decode_token_ids(targets[0].tolist()))
print("position 0 predicts:", ID_TO_TOKEN[token_ids[0, 0].item()], "->", ID_TO_TOKEN[targets[0, 0].item()])
assert token_ids.shape == targets.shape == (1, CONTEXT)
assert torch.equal(token_ids[:, 1:], targets[:, :-1])


input tokens:        <bos> The animal didn't cross the street because it was too tired
next-token targets: The animal didn't cross the street because it was too tired <eos>
position 0 predicts: <bos> -> The


## 4. Check one forward and backward pass

Cross-entropy compares the vocabulary-sized logit vector at every position with that position's shifted target. Flattening the batch and token dimensions presents all token predictions to the loss together. The assertions verify the causal mask and that gradients reach the embedding table.

In [5]:
model = TinyDecoderLM(VOCAB_SIZE, CONTEXT, D_MODEL, HEADS)
logits, weights = model(token_ids)
loss = nn.functional.cross_entropy(logits.reshape(-1, VOCAB_SIZE), targets.reshape(-1))
loss.backward()

upper_triangle = torch.triu(torch.ones(CONTEXT, CONTEXT, dtype=torch.bool), diagonal=1)
print("logits:", logits.shape, "attention:", weights.shape, "loss:", round(loss.item(), 3))
assert logits.shape == (1, CONTEXT, VOCAB_SIZE)
assert weights.shape == (1, HEADS, CONTEXT, CONTEXT)
assert torch.all(weights.masked_select(upper_triangle.unsqueeze(0).unsqueeze(0)) == 0)
assert model.token_embedding.weight.grad is not None
assert model.token_embedding.weight.grad.abs().sum() > 0


logits: torch.Size([1, 12, 13]) attention: torch.Size([1, 4, 12, 12]) loss: 2.834


## 5. Decode one next-token prediction

A language model returns logits because training needs a score for every vocabulary item. For generation, a decoding rule selects an ID and the vocabulary maps that ID back to a token. This model has random initial weights, so the result below demonstrates the decoding step rather than meaningful text generation.

In [6]:
next_token_id = logits[:, -1].argmax(dim=-1)
print("input:                     ", decode_token_ids(token_ids[0].tolist()))
print("untrained argmax next token:", ID_TO_TOKEN[next_token_id.item()])

input:                      <bos> The animal didn't cross the street because it was too tired
untrained argmax next token: The


Causal self-attention lets this model process all input positions in parallel during training without looking at future tokens. The model is decoder-only because it uses causal self-attention for autoregressive prediction and has no encoder or encoder-decoder cross-attention. The original encoder-decoder Transformer includes both an encoder stack and decoder cross-attention. Larger decoder-only language models repeat this block many times and train its parameters on many next-token prediction examples.